In [1]:
RANDOM_STATE = 42
%matplotlib inline
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "data/raw").exists() else Path.cwd().parent
assert (ROOT / "data/raw/application_record.csv").exists(), "Extraia o pacote completo antes de executar."
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from common import load, prepare, FEATURES, preprocessing
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", None)
OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / "docs/resultados"
RESULTS.mkdir(parents=True, exist_ok=True)

# 03 — Modelagem
Comparação de baseline, regressão logística e floresta aleatória, com validação por grupos.

In [2]:
from sklearn.model_selection import GroupShuffleSplit,StratifiedGroupKFold,cross_validate,cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_recall_curve
import joblib,json

d=pd.read_csv(OUT/'dataset_tratado.csv')
X=d[FEATURES];y=d.TARGET;g=d.PERFIL_ID
itr,ite=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=RANDOM_STATE).split(X,y,g))
Xtr,ytr,gtr=X.iloc[itr],y.iloc[itr],g.iloc[itr]
assert not set(gtr)&set(g.iloc[ite])
assert ytr.nunique()==2 and y.iloc[ite].nunique()==2
pd.DataFrame({'ID':d.ID,'particao':np.where(np.isin(np.arange(len(d)),itr),'treino','teste')}).to_csv(OUT/'particoes.csv',index=False)
display(pd.DataFrame({'n':[len(itr),len(ite)],'taxa':[ytr.mean(),y.iloc[ite].mean()]},index=['treino','teste']))

,n,taxa
treino,29187,0.117758
teste,7270,0.117469


GroupShuffleSplit evita perfis idênticos nos dois conjuntos, mas não garante estratificação exata. As taxas acima devem ser conferidas. Os 20% são de grupos; a proporção de linhas pode variar. O teste fica reservado até o notebook 04.

In [3]:
models={
'baseline':DummyClassifier(strategy='prior'),
'logistica':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE),
'floresta':RandomForestClassifier(n_estimators=160,min_samples_leaf=10,max_depth=14,class_weight='balanced_subsample',random_state=RANDOM_STATE,n_jobs=2)}
cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
folds=list(cv.split(Xtr,ytr,gtr))
for tr,va in folds:
    assert not set(gtr.iloc[tr])&set(gtr.iloc[va])
    assert ytr.iloc[tr].nunique()==2 and ytr.iloc[va].nunique()==2
rows=[];pipes={}
for name,clf in models.items():
    pipe=Pipeline([('preprocess',preprocessing(Xtr)),('clf',clf)])
    pipes[name]=pipe
    r=cross_validate(pipe,Xtr,ytr,cv=folds,scoring={'ap':'average_precision','auc':'roc_auc','f1':'f1','recall':'recall','precision':'precision'},n_jobs=1)
    row={'modelo':name}
    for metric in ['ap','auc','f1','recall','precision']:
        row[metric+'_media']=r['test_'+metric].mean();row[metric+'_dp']=r['test_'+metric].std()
    rows.append(row)
comparison=pd.DataFrame(rows).sort_values('ap_media',ascending=False)
comparison.to_csv(RESULTS/'validacao_cruzada.csv',index=False)
display(comparison)

/Users/italohdc/Documents/Codex/2026-10-10/10-10-26-10-39-53/work/python_deps/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/italohdc/Documents/Codex/2026-10-10/10-10-26-10-39-53/work/python_deps/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/italohdc/Documents/Codex/2026-10-10/10-10-26-10-39-53/work/python_deps/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

,modelo,ap_media,ap_dp,auc_media,auc_dp,f1_media,f1_dp,recall_media,recall_dp,precision_media,precision_dp
1,logistica,0.137034,0.001712,0.535479,0.005764,0.204826,0.000948,0.484731,0.018058,0.129916,0.001754
2,floresta,0.124004,0.003855,0.511418,0.007753,0.112458,0.012421,0.105614,0.009959,0.120389,0.015649
0,baseline,0.117758,0.000040,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Average precision (AP) resume o compromisso entre precisão e recall na classe rara e é o critério principal de seleção. ROC-AUC complementa a discriminação. F1, recall e precisão nesta tabela usam limiar 0,5. O desvio entre folds não é intervalo de confiança; diferenças pequenas não demonstram superioridade estável. O baseline quantifica o desempenho sem atributos.

In [4]:
winner=comparison.iloc[0].modelo
# Limiar escolhido exclusivamente com probabilidades out-of-fold do treino.
oof=cross_val_predict(pipes[winner],Xtr,ytr,cv=folds,method='predict_proba',n_jobs=1)[:,1]
p,r,thresholds=precision_recall_curve(ytr,oof)
f1=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
threshold=float(thresholds[np.argmax(f1)])
for name,pipe in pipes.items():
    pipe.fit(Xtr,ytr)
    joblib.dump(pipe,OUT/(name+'.joblib'))
selection={'modelo':winner,'limiar':threshold,'criterio':'AP média na CV de treino; limiar maximiza F1 OOF','target':'30+ dias'}
(RESULTS/'selecao.json').write_text(json.dumps(selection,indent=2,ensure_ascii=False))
print(selection)

{'modelo': 'logistica', 'limiar': 0.4330480082969041, 'criterio': 'AP média na CV de treino; limiar maximiza F1 OOF', 'target': '30+ dias'}


O limiar maximiza F1 como convenção acadêmica, sem estimativa de custo financeiro. O mesmo treino usado para escolher o modelo escolhe o limiar, portanto essa estimativa interna pode ser otimista; o teste externo ao processo permanece necessário. class_weight trata o desbalanceamento sem gerar exemplos sintéticos.